# Augment âm thanh (RIR + Simple noise)

Notebook tự động hóa augmentation theo mô tả trong `data/ltlong/augmented_data/README.md`:

- **RIR_method**: 20 file / mỗi file raw — định dạng `voice_rir_X_decay_Y` (Y từ 0.02 đến 0.21, bước 0.01).
- **Simple_noise**: 20 file / mỗi file raw — các loại `noise`, `stretch`, `pitch`, `shift`, `volume` (mỗi loại 4 biến thể).

Đầu ra: **`.mp3`**, mẫu **16 kHz** (khớp `TARGET_SR` trong pipeline nhận diện).

Gốc repo (`PROJECT_ROOT`): đọc từ biến môi trường **`SPEAKER_IDENTIFY_ROOT`** trong file **`.env`** (đặt cạnh `requirements.txt`), hoặc tự nhận nếu thư mục làm việc nằm trong repo (có `src/model`). Raw: `data/quocanh/raw`. Cần **ffmpeg** trong `PATH` để xuất MP3.

In [1]:
# Cài dependency (bỏ qua nếu môi trường đã có)
%pip install -q librosa soundfile numpy scipy pyroomacoustics python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import hashlib
import os
import re
import subprocess
import tempfile
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import pyroomacoustics as pra
from dotenv import load_dotenv
from scipy import signal

_ENV_KEY = "SPEAKER_IDENTIFY_ROOT"


def _resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent):
        env_file = base / ".env"
        if env_file.is_file():
            load_dotenv(env_file)
            break
    else:
        load_dotenv()

    raw = os.environ.get(_ENV_KEY, "").strip().strip('"').strip("'")
    if raw:
        return Path(raw).expanduser().resolve()

    for p in [cwd, *cwd.parents]:
        if (p / "src" / "model").is_dir():
            return p.resolve()

    raise FileNotFoundError(
        f"Không xác định được gốc repo. Tạo file .env với {_ENV_KEY}=<đường dẫn tới speaker-identify-by-voice> "
        "(xem .env.example), hoặc mở notebook khi working directory là thư mục repo / notebooks."
    )


PROJECT_ROOT = _resolve_project_root()

RAW_DIR = PROJECT_ROOT / "data" / "quocanh" / "raw"
OUT_RIR_DIR = PROJECT_ROOT / "data" / "quocanh" / "augmented" / "RIR_method"
OUT_SIMPLE_DIR = PROJECT_ROOT / "data" / "quocanh" / "augmented" / "Simple_noise"

TARGET_SR = 16000
N_AUG_PER_METHOD = 20
AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_RIR_DIR.mkdir(parents=True, exist_ok=True)
OUT_SIMPLE_DIR.mkdir(parents=True, exist_ok=True)

def _path_str(p: Path) -> str:
    return os.path.normpath(str(p.resolve()))


print("PROJECT_ROOT:", _path_str(PROJECT_ROOT))
print("RAW_DIR:", _path_str(RAW_DIR))
print("OUT_RIR_DIR:", _path_str(OUT_RIR_DIR))
print("OUT_SIMPLE_DIR:", _path_str(OUT_SIMPLE_DIR))

PROJECT_ROOT: D:\QuocAnh\Master HUST\Intro_to_DS\speaker-identify-by-voice
RAW_DIR: D:\QuocAnh\Master HUST\Intro_to_DS\speaker-identify-by-voice\data\quocanh\raw
OUT_RIR_DIR: D:\QuocAnh\Master HUST\Intro_to_DS\speaker-identify-by-voice\data\quocanh\augmented\RIR_method
OUT_SIMPLE_DIR: D:\QuocAnh\Master HUST\Intro_to_DS\speaker-identify-by-voice\data\quocanh\augmented\Simple_noise


In [3]:
def _ffmpeg_available() -> bool:
    try:
        subprocess.run(
            ["ffmpeg", "-version"],
            check=True,
            capture_output=True,
            text=True,
        )
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False


def save_mp3(y: np.ndarray, sr: int, path: Path) -> None:
    """Ghi float mono/stereo [-1, 1] ra MP3 qua WAV tạm + ffmpeg."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    y = np.asarray(y, dtype=np.float32)
    if y.ndim == 1:
        y_write = np.clip(y, -1.0, 1.0)
    else:
        y_write = np.clip(y, -1.0, 1.0)

    if not _ffmpeg_available():
        raise RuntimeError(
            "Không tìm thấy ffmpeg trong PATH. Cài ffmpeg và thêm vào PATH (Windows: winget install ffmpeg)."
        )

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        sf.write(tmp_path, y_write, sr, subtype="PCM_16")
        cmd = [
            "ffmpeg",
            "-y",
            "-i",
            tmp_path,
            "-codec:a",
            "libmp3lame",
            "-qscale:a",
            "2",
            str(path),
        ]
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"ffmpeg lỗi: {r.stderr}")
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass


def load_audio(path: Path) -> tuple[np.ndarray, int]:
    y, sr = librosa.load(str(path), sr=TARGET_SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=40)
    if y.size == 0:
        raise ValueError(f"File rỗng sau trim: {path}")
    return y.astype(np.float32), TARGET_SR


def safe_stem(name: str) -> str:
    s = Path(name).stem
    s = re.sub(r"[^\w\-]+", "_", s, flags=re.UNICODE)
    return s.strip("_") or "clip"

In [4]:
def decay_to_rt60(decay: float) -> float:
    """Ánh xạ hệ số decay (0.02…0.21) sang RT60 gần đúng (giây) cho RIR giả lập."""
    decay = float(np.clip(decay, 0.02, 0.21))
    # decay nhỏ → phòng phản xạ nhiều → RT60 dài hơn
    return float(0.25 + (0.21 - decay) * 4.0)


def rir_augment(y: np.ndarray, sr: int, decay: float, seed: int | None = None) -> np.ndarray:
    """Tích chập tín hiệu với RIR (pyroomacoustics ShoeBox + ngẫu nhiên hóa nhẹ vị trí)."""
    pra.constants.set("octave_bands_keep_dc", True)
    rng = np.random.default_rng(seed)
    rt60 = decay_to_rt60(decay)
    room_dim = np.array(
        [3.0 + rng.uniform(-0.4, 0.4), 4.0 + rng.uniform(-0.5, 0.5), 2.5 + rng.uniform(-0.2, 0.2)]
    )
    try:
        e_abs, max_order = pra.inverse_sabine(rt60, room_dim)
    except Exception:
        e_abs, max_order = 0.25, 12
    materials = pra.Material(e_abs)
    room = pra.ShoeBox(
        room_dim,
        fs=sr,
        materials=materials,
        max_order=min(int(max_order), 15),
        use_rand_ism=True,
    )
    src = np.array([rng.uniform(0.5, room_dim[0] - 0.5), rng.uniform(0.5, room_dim[1] - 0.5), 1.2])
    mic = np.array([rng.uniform(0.5, room_dim[0] - 0.5), rng.uniform(0.5, room_dim[1] - 0.5), 1.5])
    room.add_source(src)
    room.add_microphone(mic)
    room.compute_rir()
    rir = np.asarray(room.rir[0][0], dtype=np.float32)
    if rir.size == 0:
        raise RuntimeError("RIR rỗng")
    out = signal.fftconvolve(y, rir, mode="full")[: y.size + int(0.3 * sr)]
    peak = np.max(np.abs(out)) + 1e-8
    if peak > 1.0:
        out = out / peak
    return out.astype(np.float32)


def mix_noise_snr(clean: np.ndarray, snr_db: float, rng: np.random.Generator) -> np.ndarray:
    noise = rng.standard_normal(clean.shape).astype(np.float32)
    p_clean = np.mean(clean**2) + 1e-8
    p_noise = np.mean(noise**2) + 1e-8
    snr = 10 ** (snr_db / 10)
    scale = np.sqrt(p_clean / (p_noise * snr))
    return np.clip(clean + scale * noise, -1.0, 1.0)


def simple_augmentations(y: np.ndarray, sr: int, rng: np.random.Generator) -> list[tuple[str, np.ndarray]]:
    """Trả về đúng 20 cặp (type_label, audio) theo README: noise, stretch, pitch, shift, volume × 4."""
    out: list[tuple[str, np.ndarray]] = []

    snrs = (15.0, 20.0, 25.0, 30.0)
    for snr in snrs:
        out.append(("noise", mix_noise_snr(y, snr, rng)))

    rates = (0.88, 0.92, 1.08, 1.12)
    for r in rates:
        ys = librosa.effects.time_stretch(y, rate=float(r))
        out.append(("stretch", ys.astype(np.float32)))

    n_steps_list = (-2.0, -1.0, 1.0, 2.0)
    for ns in n_steps_list:
        yp = librosa.effects.pitch_shift(y, sr=sr, n_steps=float(ns))
        out.append(("pitch", yp.astype(np.float32)))

    shifts_frac = (-0.03, -0.015, 0.015, 0.03)
    for frac in shifts_frac:
        n = int(round(frac * len(y)))
        out.append(("shift", np.roll(y, n).astype(np.float32)))

    gains_db = (-6.0, -3.0, 3.0, 6.0)
    for gdb in gains_db:
        g = 10 ** (gdb / 20)
        out.append(("volume", np.clip(y * g, -1.0, 1.0).astype(np.float32)))

    assert len(out) == 20
    return out

In [5]:
def iter_raw_files(raw_dir: Path):
    raw_dir = Path(raw_dir)
    if not raw_dir.is_dir():
        raise FileNotFoundError(
            f"Không có thư mục raw: {raw_dir}\n"
            "  → Tạo `data/quocanh/raw` hoặc kiểm tra SPEAKER_IDENTIFY_ROOT trong .env."
        )
    for p in sorted(raw_dir.rglob("*")):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS:
            yield p


def process_file(raw_path: Path) -> None:
    stem = safe_stem(raw_path.name)
    y, sr = load_audio(raw_path)
    seed_u32 = int(hashlib.md5(stem.encode("utf-8")).hexdigest()[:8], 16)
    rng = np.random.default_rng(seed_u32)

    # --- RIR: X=1..20, Y = 0.02 + (X-1)*0.01
    for x in range(1, N_AUG_PER_METHOD + 1):
        decay = round(0.02 + (x - 1) * 0.01, 2)
        y_rir = rir_augment(y, sr, decay, seed=rng.integers(0, 2**31 - 1))
        name = f"{stem}_voice_rir_{x}_decay_{decay:.2f}.mp3"
        save_mp3(y_rir, sr, OUT_RIR_DIR / name)

    # --- Simple noise / stretch / pitch / shift / volume
    simple_list = simple_augmentations(y, sr, rng)
    for idx, (kind, y_aug) in enumerate(simple_list, start=1):
        name = f"{stem}_aug_{idx}_{kind}.mp3"
        save_mp3(y_aug, sr, OUT_SIMPLE_DIR / name)

    print(f"OK: {raw_path.name} -> {N_AUG_PER_METHOD} RIR + {N_AUG_PER_METHOD} simple")


def main():
    files = list(iter_raw_files(RAW_DIR))
    if not files:
        exts = ", ".join(sorted(AUDIO_EXTENSIONS))
        print(f"Không có file âm thanh trong {_path_str(RAW_DIR)} (đuôi: {exts}).")
        return
    print(f"Tìm thấy {len(files)} file. Bắt đầu augment…")
    for fp in files:
        try:
            process_file(fp)
        except Exception as e:
            print(f"Lỗi {fp}: {e}")


main()

Tìm thấy 10 file. Bắt đầu augment…
OK: Bao_Hue.mp3 -> 20 RIR + 20 simple
OK: Duc.mp3 -> 20 RIR + 20 simple
OK: Hai_Yen.mp3 -> 20 RIR + 20 simple
OK: Hung.mp3 -> 20 RIR + 20 simple
OK: Lien.mp3 -> 20 RIR + 20 simple
OK: Linh_Dao.mp3 -> 20 RIR + 20 simple
OK: Nam_Minh.mp3 -> 20 RIR + 20 simple
OK: Nguyen_Lan.mp3 -> 20 RIR + 20 simple
OK: Trung.mp3 -> 20 RIR + 20 simple
OK: Van_Son.mp3 -> 20 RIR + 20 simple
